# Phase 2d / 3 — KD-LoRA vs SFT-LoRA: In-Domain + Cross-Domain Evaluation

**Goal:** Run all four models on CNN/DailyMail (in-domain, training distribution), XSum (cross-domain, extreme news summaries), and SAMSum (cross-domain, dialogue). Compare KD-LoRA vs SFT-LoRA in both settings to answer the headline research question and test generalization.

**Models:**
1. `Qwen/Qwen2.5-7B-Instruct` — teacher (ceiling reference)
2. `Qwen/Qwen2.5-0.5B` — untrained student baseline (floor reference, same as Phase 1)
3. `Qwen/Qwen2.5-0.5B + KD-LoRA` — your KD adapter from the Hub
4. `Qwen/Qwen2.5-0.5B + SFT-LoRA` — your SFT adapter from the Hub

**Approach:**
- Adapters are pulled from the Hub, **merged into the base model** via `merge_and_unload()`, saved to a temp directory, then loaded into vLLM as a normal full model.
- One vLLM engine in VRAM at a time; aggressive teardown between models.
- Same generation config across all four models per dataset, so ROUGE numbers are directly comparable.

**Hardware:** Single 40GB GPU (works on 80GB too).

## 1. Install dependencies

Same pinned versions that worked for Phase 1 + training.

In [1]:
!pip install -q "numpy<2.0" \
    "vllm==0.6.3" \
    "transformers==4.46.0" \
    "huggingface_hub>=0.25.0,<0.27.0" \
    "tokenizers>=0.20,<0.21" \
    "peft==0.13.2" \
    "bert-score==0.3.13" \
    datasets==2.21.0 evaluate==0.4.3 rouge_score==0.1.2 sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Reason for being yanked: This version unfortunately does not work with 3.8 but we did not drop the support yet
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.5/193.5 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 135.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.1 MB/s eta 0:00

## 2. Imports, HF login, config

Set `HF_USERNAME` to your Hugging Face username and double-check the adapter repo names match what you actually pushed.

In [9]:
import gc, json, re, shutil, tempfile, time
import os
from pathlib import Path

# Cache HF models (incl. roberta-large for BERTScore) so repeat runs are fast
os.environ.setdefault('HF_HOME', '/content/hf_cache')

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from datasets import load_dataset
from vllm import LLM, SamplingParams
from vllm.distributed.parallel_state import destroy_model_parallel
import evaluate
from bert_score import score as bertscore_compute

# Auth for private adapters
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'))
    print('HF login OK')
except Exception as e:
    print(f'(Skipping HF login: {e})')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ---- EDIT THESE ----------------------------------------------------------
HF_USERNAME = 'Harsha901'
KD_REPO  = f'{HF_USERNAME}/qwen2.5-0.5b-kd-lora-cnndm-50k'
SFT_REPO = f'{HF_USERNAME}/qwen2.5-0.5b-sft-lora-cnndm-50k'
# --------------------------------------------------------------------------

CONFIG = {
    'teacher_model': 'Qwen/Qwen2.5-7B-Instruct',
    'student_base':  'Qwen/Qwen2.5-0.5B',
    'kd_adapter':    KD_REPO,
    'sft_adapter':   SFT_REPO,

    'num_eval_samples': 1000,
    'temperature': 0.0,
    'seed': 42,
    'results_dir': './eval_results_v2',     # new dir so v1 results are preserved

    # vLLM
    'vllm_gpu_memory_utilization': 0.85,
    'vllm_max_num_seqs': 128,
    'vllm_max_num_batched_tokens': 16384,

    # ----- DEGENERATION FIXES (new in v2) -------------------------------
    'repetition_penalty': 1.15,            # >1 discourages reusing tokens
    'frequency_penalty':  0.3,             # discourages high-frequency tokens
    # Stop strings: cut generation when these appear. List was built from
    # observed degeneration patterns in v1 outputs.
    'stop_strings': [
        '"""",',          # the JSON-string-comma loop seen in KD
        '\n\nArticle:', '\n\nConversation:', '\n\nDocument:',   # new-section headers
        '\n\nSummary:',                                           # restart attempts
        '游戏副本', '핮', '겁', 'norge', 'LOCK:', '(INVOKE',        # specific garbage tokens
    ],
    # --------------------------------------------------------------------

    'eval_teacher': True,
    'eval_baseline_student': True,
    'eval_kd': True,
    'eval_sft': True,
}
Path(CONFIG['results_dir']).mkdir(exist_ok=True)
torch.manual_seed(CONFIG['seed'])
print(json.dumps({k: v for k, v in CONFIG.items() if not k.startswith('eval_')}, indent=2))

HF login OK
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB
{
  "teacher_model": "Qwen/Qwen2.5-7B-Instruct",
  "student_base": "Qwen/Qwen2.5-0.5B",
  "kd_adapter": "Harsha901/qwen2.5-0.5b-kd-lora-cnndm-50k",
  "sft_adapter": "Harsha901/qwen2.5-0.5b-sft-lora-cnndm-50k",
  "num_eval_samples": 1000,
  "temperature": 0.0,
  "seed": 42,
  "results_dir": "./eval_results_v2",
  "vllm_gpu_memory_utilization": 0.85,
  "vllm_max_num_seqs": 128,
  "vllm_max_num_batched_tokens": 16384,
  "repetition_penalty": 1.15,
  "frequency_penalty": 0.3,
  "stop_strings": [
    "\"\"\"\",",
    "\n\nArticle:",
    "\n\nConversation:",
    "\n\nDocument:",
    "\n\nSummary:",
    "\u6e38\u620f\u526f\u672c",
    "\ud56e",
    "\uac81",
    "norge",
    "LOCK:",
    "(INVOKE"
  ]
}


## 3. Dataset configs

Each dataset has its own input/target columns and a dataset-appropriate prompt. Output length targets match the typical summary length in that corpus.

In [10]:
DATASETS = {
    'cnn_dailymail': {
        'hf_name': 'cnn_dailymail',
        'hf_config': '3.0.0',
        'split': 'test',
        'input_col': 'article',
        'target_col': 'highlights',
        'doc_type': 'news article',
        'system_prompt': (
            'You are a concise news summarizer. Write a short summary of the article in 2-3 sentences. '
            'Output only the summary itself, with no preamble, headers, or commentary.'
        ),
        'user_template': 'Article:\n{text}\n\nSummary:',
        'max_input_tokens': 3000,
        'max_new_tokens': 160,
        'in_domain': True,
    },
    'xsum': {
        'hf_name': 'EdinburghNLP/xsum',
        'hf_config': None,
        'split': 'test',
        'input_col': 'document',
        'target_col': 'summary',
        'doc_type': 'BBC news article',
        'system_prompt': (
            'You are a concise summarizer. Write a single sentence that captures the main point of the article. '
            'Output only the sentence itself, with no preamble or commentary.'
        ),
        'user_template': 'Article:\n{text}\n\nSummary:',
        'max_input_tokens': 2000,
        'max_new_tokens': 64,
        'in_domain': False,
    },
    'samsum': {
        'hf_name': 'knkarthick/samsum',
        'hf_config': None,
        'split': 'test',
        'input_col': 'dialogue',
        'target_col': 'summary',
        'doc_type': 'messenger conversation',
        'system_prompt': (
            'You are a concise summarizer of messenger-style conversations. '
            'Write a brief 1-2 sentence summary describing what was discussed. '
            'Output only the summary itself, with no preamble or commentary.'
        ),
        'user_template': 'Conversation:\n{text}\n\nSummary:',
        'max_input_tokens': 1024,
        'max_new_tokens': 80,
        'in_domain': False,
    },
    'dialogsum': {
    'hf_name': 'knkarthick/dialogsum',
    'hf_config': None,
    'split': 'test',
    'input_col': 'dialogue',
    'target_col': 'summary',
    'doc_type': 'conversation',
    'system_prompt': (
        'You are a concise summarizer of conversations. '
        'Write a brief 1-3 sentence summary of what was discussed. '
        'Output only the summary itself, with no preamble or commentary.'
    ),
    'user_template': 'Conversation:\n{text}\n\nSummary:',
    'max_input_tokens': 1024,
    'max_new_tokens': 80,
    'in_domain': False,
},
}

def load_eval_split(name: str, n: int | None):
    d = DATASETS[name]
    ds = (load_dataset(d['hf_name'], d['hf_config'], split=d['split'])
          if d['hf_config'] else load_dataset(d['hf_name'], split=d['split']))
    if n is not None:
        ds = ds.shuffle(seed=CONFIG['seed']).select(range(min(n, len(ds))))
    return ds

# Peek
for name in DATASETS:
    ds = load_eval_split(name, 1)
    d = DATASETS[name]
    print(f"\n=== {name} (in-domain={d['in_domain']}) ===")
    print(f"  input  ({d['input_col']}): {ds[0][d['input_col']][:200]}...")
    print(f"  target ({d['target_col']}): {ds[0][d['target_col']]}")


=== cnn_dailymail (in-domain=True) ===
  input  (article): (CNN) I see signs of a revolution everywhere. I see it in the op-ed pages of the newspapers, and on the state ballots in nearly half the country. I see it in politicians who once preferred to play it ...
  target (highlights): CNN's Dr. Sanjay Gupta says we should legalize medical marijuana now .
He says he knows how easy it is do nothing "because I did nothing for too long"

=== xsum (in-domain=False) ===
  input  (document): Sarah Johnson was one of 21 women heading to Liverpool when their minibus was hit by a lorry on the M62.
Her friend Bethany Jones, 18, was killed while Ms Johnson and several others were badly hurt.
M...
  target (summary): A woman who was seriously hurt in a fatal hen party motorway crash is now helping other major trauma victims rebuild their lives.

=== samsum (in-domain=False) ===
  input  (dialogue): Claire: <file_photo>
Kim: Looks delicious...
Linda: No way... Look what I'm cooking right now:
Linda

## 4. Prompt builder & post-processing

Truncate the *document* (not the whole chat prompt) so the generation marker is preserved. Strip preambles like "Here is a summary:" before scoring.

In [11]:
PREAMBLE_RE = re.compile(
    r'^\s*(here(?:\s+is|\'s)?\s+(?:a\s+)?(?:brief\s+|short\s+|concise\s+)?summary[:\s\-]*|'
    r'summary[:\s\-]+|'
    r'the\s+(?:article|conversation|document)\s+(?:is\s+about|discusses|describes)[:\s\-]*)',
    re.IGNORECASE,
)

# Detects long runs of non-ASCII chars — usually a degeneration tail
NON_ASCII_TAIL_RE = re.compile(r'[^\x00-\x7f]{2,}.*$', re.DOTALL)

# Detects identical-sentence-repeated-3+-times (the SFT failure mode)
def trim_repeated_sentences(text: str) -> str:
    """If the same sentence appears 3+ times in a row, keep only the first."""
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    if len(parts) < 4:
        return text
    out = []
    prev = None
    repeat_count = 0
    for s in parts:
        norm = s.strip().lower()
        if norm == prev:
            repeat_count += 1
            if repeat_count >= 2:   # third+ occurrence, stop here
                break
        else:
            repeat_count = 0
            prev = norm
        out.append(s)
    return ' '.join(out)

def clean_prediction(text: str) -> tuple[str, bool]:
    """Aggressive cleanup: strip preamble, cut non-ASCII tails, dedupe repetitions."""
    original = text.strip()
    # 1. strip preamble
    s = PREAMBLE_RE.sub('', original).strip().lstrip('\n').strip()
    # 2. cut everything from the first sustained non-ASCII run onward
    s = NON_ASCII_TAIL_RE.sub('', s).strip()
    # 3. dedupe repeated sentences
    s = trim_repeated_sentences(s)
    # 4. final whitespace tidy
    s = re.sub(r'\s+\n', '\n', s).strip()
    return s, (s != original)

def build_prompt(tokenizer, text: str, ds_cfg: dict) -> str:
    empty_msgs = [
        {'role': 'system', 'content': ds_cfg['system_prompt']},
        {'role': 'user', 'content': ds_cfg['user_template'].format(text='')},
    ]
    overhead_ids = tokenizer.apply_chat_template(empty_msgs, tokenize=True, add_generation_prompt=True)
    max_text_tokens = max(128, ds_cfg['max_input_tokens'] - len(overhead_ids) - 8)

    text_ids = tokenizer(
        text, add_special_tokens=False, truncation=True, max_length=max_text_tokens
    ).input_ids
    trimmed = tokenizer.decode(text_ids, skip_special_tokens=True)

    msgs = [
        {'role': 'system', 'content': ds_cfg['system_prompt']},
        {'role': 'user', 'content': ds_cfg['user_template'].format(text=trimmed)},
    ]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

rouge = evaluate.load('rouge')

## 5. Merge-and-save helper

Loads a LoRA adapter onto the base model, merges weights, and saves the full model to a temp directory. vLLM then loads that temp directory like any normal model. Temp dirs are deleted after each eval to free disk.

In [12]:
def merge_adapter_to_tempdir(base_model_name: str, adapter_repo: str) -> str:
    tmp = tempfile.mkdtemp(prefix='merged_', dir='.')
    print(f'  Loading base model {base_model_name}...')
    base = AutoModelForCausalLM.from_pretrained(base_model_name, torch_dtype=torch.bfloat16)
    print(f'  Attaching adapter from {adapter_repo}...')
    merged = PeftModel.from_pretrained(base, adapter_repo)
    print(f'  Merging weights...')
    merged = merged.merge_and_unload()
    print(f'  Saving merged model to {tmp}...')
    merged.save_pretrained(tmp, safe_serialization=True)
    tok = AutoTokenizer.from_pretrained(adapter_repo)  # adapter repos include tokenizer
    tok.save_pretrained(tmp)
    del merged, base, tok
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return tmp

def teardown_vllm(llm) -> None:
    try:
        destroy_model_parallel()
    except Exception as e:
        print(f'  destroy_model_parallel warning: {e}')
    try:
        del llm.llm_engine.model_executor
    except Exception:
        pass
    del llm
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

## 6. Generic evaluation function

Takes a model name (path or repo), evaluates it on every dataset in `DATASETS`, returns a list of per-dataset result dicts. One vLLM engine load per model, then iterates over the three datasets in-memory.

In [13]:
def evaluate_model_on_all_datasets(label: str, model_path: str) -> list[dict]:
    print(f'\n{"="*70}\n{label}  ({model_path})\n{"="*70}')

    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    max_model_len = max(d['max_input_tokens'] + d['max_new_tokens'] for d in DATASETS.values())

    llm = LLM(
        model=model_path,
        dtype='bfloat16',
        tensor_parallel_size=1,
        trust_remote_code=True,
        gpu_memory_utilization=CONFIG['vllm_gpu_memory_utilization'],
        max_model_len=max_model_len,
        max_num_seqs=CONFIG['vllm_max_num_seqs'],
        max_num_batched_tokens=CONFIG['vllm_max_num_batched_tokens'],
        enable_prefix_caching=True,
        seed=CONFIG['seed'],
    )
    print('  vLLM engine loaded.')

    per_dataset_results = []
    for ds_name, ds_cfg in DATASETS.items():
        print(f'\n  -- {ds_name} --')
        eval_ds = load_eval_split(ds_name, CONFIG['num_eval_samples'])
        texts = eval_ds[ds_cfg['input_col']]
        refs = eval_ds[ds_cfg['target_col']]
        prompts = [build_prompt(tokenizer, t, ds_cfg) for t in texts]

        sampling = SamplingParams(
            temperature=CONFIG['temperature'],
            top_p=1.0,
            max_tokens=ds_cfg['max_new_tokens'],
            skip_special_tokens=True,
            repetition_penalty=CONFIG['repetition_penalty'],
            frequency_penalty=CONFIG['frequency_penalty'],
            stop=CONFIG['stop_strings'],
        )

        start = time.time()
        outs = llm.generate(prompts, sampling, use_tqdm=True)
        total = time.time() - start

        raw_preds, clean_preds, had_pre = [], [], []
        finish_reasons = {}
        for o in outs:
            raw = o.outputs[0].text
            fr = o.outputs[0].finish_reason
            finish_reasons[fr] = finish_reasons.get(fr, 0) + 1
            c, h = clean_prediction(raw)
            raw_preds.append(raw.strip()); clean_preds.append(c); had_pre.append(h)
        print(f'    Finish reasons: {finish_reasons}')

        # --- ROUGE ----------------------------------------------------------
        sc = rouge.compute(predictions=clean_preds, references=refs, use_stemmer=True)
        sc_raw = rouge.compute(predictions=raw_preds, references=refs, use_stemmer=True)

        # --- BERTScore ------------------------------------------------------
        print(f'    Computing BERTScore on {len(clean_preds)} predictions...')
        bs_t0 = time.time()
        bert_P, bert_R, bert_F = bertscore_compute(
            cands=clean_preds, refs=refs,
            lang='en', rescale_with_baseline=True,
            verbose=False, batch_size=64,
        )
        bertscore_f1 = float(bert_F.mean()) * 100
        bertscore_precision = float(bert_P.mean()) * 100
        bertscore_recall = float(bert_R.mean()) * 100
        print(f'    BERTScore done in {time.time() - bs_t0:.1f}s  (F1={bertscore_f1:.2f})')

        res = {
            'model_label': label,
            'model_path': model_path,
            'dataset': ds_name,
            'in_domain': ds_cfg['in_domain'],
            'num_samples': len(texts),
            'gen_time_s': round(total, 1),
            'samples_per_sec': round(len(texts)/total, 3),
            'rouge1_clean': round(sc['rouge1']*100, 3),
            'rouge2_clean': round(sc['rouge2']*100, 3),
            'rougeL_clean': round(sc['rougeL']*100, 3),
            'rougeLsum_clean': round(sc['rougeLsum']*100, 3),
            'rouge1_raw': round(sc_raw['rouge1']*100, 3),
            'bertscore_f1': round(bertscore_f1, 3),
            'bertscore_precision': round(bertscore_precision, 3),
            'bertscore_recall': round(bertscore_recall, 3),
            'avg_pred_words': round(sum(len(p.split()) for p in clean_preds)/len(clean_preds), 1),
            'avg_ref_words': round(sum(len(r.split()) for r in refs)/len(refs), 1),
            'preamble_rate': round(sum(had_pre)/len(had_pre), 3),
            'finish_reasons': finish_reasons,
        }
        print('    ' + ' | '.join(f'{k}={v}' for k, v in res.items() if k in (
            'rouge1_clean', 'rougeL_clean', 'bertscore_f1', 'avg_pred_words', 'preamble_rate')))

        safe = f"{label}__{ds_name}".replace(' ', '_').replace('/', '_')
        with open(Path(CONFIG['results_dir']) / f'samples_{safe}.json', 'w') as f:
            json.dump({
                'result': res,
                'samples': [{'reference': refs[i], 'raw': raw_preds[i], 'clean': clean_preds[i]}
                            for i in range(min(20, len(refs)))],
            }, f, indent=2)
        per_dataset_results.append(res)

    teardown_vllm(llm)
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return per_dataset_results

## 7. Run the four models

For each LoRA model: merge adapter into base, save to temp dir, eval, clean up temp dir.

In [14]:
all_results = []
temp_dirs = []  # track for cleanup

if CONFIG['eval_teacher']:
    all_results.extend(evaluate_model_on_all_datasets('teacher_7B', CONFIG['teacher_model']))

if CONFIG['eval_baseline_student']:
    all_results.extend(evaluate_model_on_all_datasets('student_baseline_0.5B', CONFIG['student_base']))

if CONFIG['eval_kd']:
    print('\n>>> Merging KD adapter into base...')
    kd_path = merge_adapter_to_tempdir(CONFIG['student_base'], CONFIG['kd_adapter'])
    temp_dirs.append(kd_path)
    all_results.extend(evaluate_model_on_all_datasets('student_KD_LoRA', kd_path))
    # Optional: free disk now if you want — keeping for inspection is fine on Colab
    # shutil.rmtree(kd_path); temp_dirs.remove(kd_path)

if CONFIG['eval_sft']:
    print('\n>>> Merging SFT adapter into base...')
    sft_path = merge_adapter_to_tempdir(CONFIG['student_base'], CONFIG['sft_adapter'])
    temp_dirs.append(sft_path)
    all_results.extend(evaluate_model_on_all_datasets('student_SFT_LoRA', sft_path))

print(f'\nCompleted {len(all_results)} (model, dataset) eval pairs.')


teacher_7B  (Qwen/Qwen2.5-7B-Instruct)
INFO 05-17 19:04:59 llm_engine.py:237] Initializing an LLM engine (vdev) with config: model='Qwen/Qwen2.5-7B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=3160, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=42, served_model_name=Qwen/Qwen2.5-7B-Instruct, use_v2_block_manager=True, num_scheduler_steps=1, chunked_pr

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


INFO 05-17 19:05:07 model_runner.py:1071] Loading model weights took 14.2409 GB
INFO 05-17 19:05:09 gpu_executor.py:122] # GPU blocks: 59410, # CPU blocks: 4681
INFO 05-17 19:05:09 gpu_executor.py:126] Maximum concurrency for 3160 tokens per request: 300.81x
INFO 05-17 19:05:09 model_runner.py:1402] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 05-17 19:05:09 model_runner.py:1406] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 05-17 19:05:28 model_runner.py:1530] Graph capturing finished in 19 secs.
  vLLM engine loaded.

  -- cnn_dailymail --


Processed prompts: 100%|██████████| 1000/1000 [01:09<00:00, 14.37it/s, est. speed input: 13120.01 toks/s, output: 820.43 toks/s]


    Finish reasons: {'stop': 1000}
    Computing BERTScore on 1000 predictions...


...

    BERTScore done in 4.6s  (F1=16.31)
    rouge1_clean=24.307 | rougeL_clean=16.973 | bertscore_f1=16.307 | avg_pred_words=41.7 | preamble_rate=0.149

  -- dialogsum --


Processed prompts: 100%|██████████| 1000/1000 [00:14<00:00, 70.21it/s, est. speed input: 17485.44 toks/s, output: 3537.99 toks/s]


    Finish reasons: {'stop': 844, 'length': 156}
    Computing BERTScore on 1000 predictions...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


    BERTScore done in 4.4s  (F1=6.91)
    rouge1_clean=19.758 | rougeL_clean=14.505 | bertscore_f1=6.911 | avg_pred_words=42.5 | preamble_rate=0.09

Completed 16 (model, dataset) eval pairs.


## 8. Cleanup temp merged models

In [ ]:
for d in temp_dirs:
    try:
        shutil.rmtree(d)
        print(f'Removed {d}')
    except Exception as e:
        print(f'Could not remove {d}: {e}')

## 9. Headline tables

Pivot the results so each row is a model and each column group is a dataset. Compute the KD—SFT delta per dataset — that's the publication-relevant number.

In [15]:
import pandas as pd

df = pd.DataFrame(all_results)
# Drop the finish_reasons dict so CSV is clean
df_csv = df.drop(columns=['finish_reasons'], errors='ignore')
df_csv.to_csv(Path(CONFIG['results_dir']) / 'all_results.csv', index=False)
with open(Path(CONFIG['results_dir']) / 'all_results.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=str)

pivot = df.pivot_table(
    index='model_label',
    columns='dataset',
    values=['rouge1_clean', 'rouge2_clean', 'rougeL_clean', 'bertscore_f1', 'avg_pred_words'],
    aggfunc='first',
)
print('\n=== Full results (ROUGE + BERTScore + length) ===')
print(pivot.to_string())
pivot.to_csv(Path(CONFIG['results_dir']) / 'pivot_full.csv')

order = ['teacher_7B', 'student_baseline_0.5B', 'student_SFT_LoRA', 'student_KD_LoRA']

r1 = df.pivot_table(index='model_label', columns='dataset', values='rouge1_clean', aggfunc='first')
r1 = r1.reindex([x for x in order if x in r1.index])
print('\n=== ROUGE-1 by model x dataset ===')
print(r1.to_string())
r1.to_csv(Path(CONFIG['results_dir']) / 'headline_rouge1.csv')

bs = df.pivot_table(index='model_label', columns='dataset', values='bertscore_f1', aggfunc='first')
bs = bs.reindex([x for x in order if x in bs.index])
print('\n=== BERTScore-F1 by model x dataset ===')
print(bs.to_string())
bs.to_csv(Path(CONFIG['results_dir']) / 'headline_bertscore.csv')

# Length sanity check — did the fixes work?
lens = df.pivot_table(index='model_label', columns='dataset', values='avg_pred_words', aggfunc='first')
lens = lens.reindex([x for x in order if x in lens.index])
print('\n=== avg_pred_words (lower = degeneration controlled) ===')
print(lens.to_string())

if 'student_KD_LoRA' in r1.index and 'student_SFT_LoRA' in r1.index:
    delta_r1 = r1.loc['student_KD_LoRA'] - r1.loc['student_SFT_LoRA']
    delta_bs = bs.loc['student_KD_LoRA'] - bs.loc['student_SFT_LoRA']

    print('\n=== KD-LoRA — SFT-LoRA delta (positive = KD wins) ===')
    print(f'{"dataset":<18} {"type":<14} {"ΔROUGE-1":>10} {"ΔBERTScore":>12}   verdict')
    print('-' * 80)
    for ds_name in delta_r1.index:
        cfg = DATASETS[ds_name]
        tag = 'in-domain' if cfg['in_domain'] else 'cross-domain'
        dr = delta_r1[ds_name]
        db = delta_bs[ds_name]
        rouge_verdict = 'KD' if dr > 0.5 else ('SFT' if dr < -0.5 else 'tie')
        bert_verdict  = 'KD' if db > 0.3 else ('SFT' if db < -0.3 else 'tie')
        if rouge_verdict == bert_verdict:
            verdict = f'{rouge_verdict} (both agree)'
        else:
            verdict = f'ROUGE→{rouge_verdict}, BERT→{bert_verdict} (disagree)'
        print(f'{ds_name:<18} {tag:<14} {dr:>+10.3f} {db:>+12.3f}   {verdict}')

if 'student_baseline_0.5B' in r1.index:
    print('\n=== Training lift vs untrained student ===')
    for trained in ('student_KD_LoRA', 'student_SFT_LoRA'):
        if trained in r1.index:
            print(f'\n  {trained}:')
            for ds_name in r1.columns:
                lift_r1 = r1.loc[trained, ds_name] - r1.loc['student_baseline_0.5B', ds_name]
                lift_bs = bs.loc[trained, ds_name] - bs.loc['student_baseline_0.5B', ds_name]
                print(f'    {ds_name:<18}: ΔROUGE-1 = {lift_r1:+.3f}   ΔBERTScore = {lift_bs:+.3f}')


=== Full results (ROUGE + BERTScore + length) ===
                      avg_pred_words                         bertscore_f1                            rouge1_clean                            rouge2_clean                           rougeL_clean                          
dataset                cnn_dailymail dialogsum samsum  xsum cnn_dailymail dialogsum  samsum    xsum cnn_dailymail dialogsum  samsum    xsum cnn_dailymail dialogsum  samsum   xsum cnn_dailymail dialogsum  samsum    xsum
model_label                                                                                                                                                                                                               
student_KD_LoRA                 50.1      41.6   37.9  45.9        19.824    16.783  27.080  21.368        31.359    21.496  26.122  21.461         8.992     4.181   5.061  3.921        19.777    15.303  18.292  14.383
student_SFT_LoRA                70.2      42.5   41.7  44.2        24.074

## How to read the headline tables

- **In-domain (CNN/DailyMail):** the direct answer to "does KD beat SFT for the training distribution?" Bigger delta = stronger headline.
- **Cross-domain (XSum, SAMSum):** does the advantage *transfer*? If KD wins in-domain AND on both held-out datasets, that's the strongest possible result — distillation imparts genuine capability rather than just memorized output patterns. If KD wins in-domain but SFT wins out-of-domain (or vice-versa), that's its own interesting finding to discuss.
- **Training lift:** sanity check that LoRA training did *anything*. If both trained models are within ~1 point of the untrained baseline, training failed somewhere.

## Next steps
If the headline gap is positive and consistent across datasets, you have a paper. Scale up training to 50k examples for final numbers, add LLM-judge or BERTScore as a second metric (ROUGE alone is thin for instruction-tuned LLMs), and write up.